# Обработка данных в Pandas

Практический ноутбук к модулю **«Обработка данных в Pandas»** программы **«Специалист по большим данным»**.

В ноутбуке используется единый учебный набор данных о котах:

- `cats_raw.csv` — основная «сырая» таблица;
- `Данные по котам для Pandas.xlsx` — Excel-файл с листами `cats_raw`, `breeds`, `visits`, `README`, `data_dictionary`.

Цель: пройти полный цикл работы с данными — **загрузка → диагностика → очистка → преобразование → анализ → объединение → агрегация → экспорт**.


## 1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print('Pandas:', pd.__version__)

Pandas: 3.0.3


## 2. Пути к файлам

Если ноутбук лежит в одной папке с данными, менять ничего не нужно.  
Для запуска в другой среде достаточно изменить `DATA_DIR`.


In [2]:
DATA_DIR = Path(".")
CSV_PATH = DATA_DIR / "cats_raw.csv"
XLSX_PATH = DATA_DIR / "Данные по котам для Pandas.xlsx"

print('CSV:', CSV_PATH)
print('XLSX:', XLSX_PATH)

CSV: cats_raw.csv
XLSX: Данные по котам для Pandas.xlsx


## 3. Чтение CSV

In [3]:
cats = pd.read_csv(CSV_PATH)
cats.head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021


### Размер таблицы

In [4]:
cats.shape

(125, 12)

В исходном CSV должно быть **125 строк и 12 столбцов**.

Посмотрим названия полей.


In [6]:
cats.columns.tolist()

['cat_id',
 'name',
 'breed',
 'sex',
 'color',
 'birth_date',
 'weight_kg',
 'height_cm',
 'city',
 'vaccinated',
 'sterilized',
 'owner_id']

## 4. Чтение Excel

In [7]:
excel = pd.ExcelFile(XLSX_PATH)

excel.sheet_names

['README', 'cats_raw', 'breeds', 'visits', 'data_dictionary']

### Загружаем отдельные листы

In [8]:
cats_excel = pd.read_excel(XLSX_PATH, sheet_name='cats_raw')
breeds = pd.read_excel(XLSX_PATH, sheet_name='breeds')
visits = pd.read_excel(XLSX_PATH, sheet_name='visits')

print('cats_raw', cats_excel.shape)
print('breeds', breeds.shape)
print('visits', visits.shape)

cats_raw (125, 12)
breeds (12, 6)
visits (210, 7)


## 5. Первичное знакомство с DataFrame

In [9]:
cats.head(n=7)

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
6,CAT007,Ириска,Бенгальская,кот,белый,2018-05-07,"4,9",32.0,Новосибирск,да,нет,OWN031


In [10]:
cats.tail(n=7)

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
118,CAT119,Тиша,Русская голубая,кот,черепаховый,2024-12-26,4.5,30.0,Томск,Нет,да,OWN010
119,CAT120,Том,Норвежская лесная,кошка,голубой,2021-03-12,6.9,37.0,Москва,НЕТ,нет,OWN015
120,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
121,CAT019,Кира,Бенгальская,кот,рыжий,2024-04-12,4.4,30.0,Новосибирск,да,да,OWN008
122,CAT055,Мия,Бенгальская,кот,рыжий,2020-01-04,6.2,33.0,Томск,да,нет,OWN022
123,CAT078,Тиша,Канадский сфинкс,кошка,табби,2017-03-20,5.2,33.0,Томск,НЕТ,да,OWN054
124,CAT104,Буся,Персидская,кошка,серебристый,2016-07-15,4.3,29.0,Красноярск,Да,да,OWN018


### Структура таблицы

In [11]:
cats.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cat_id      125 non-null    str    
 1   name        125 non-null    str    
 2   breed       125 non-null    str    
 3   sex         125 non-null    str    
 4   color       118 non-null    str    
 5   birth_date  125 non-null    str    
 6   weight_kg   117 non-null    str    
 7   height_cm   123 non-null    float64
 8   city        125 non-null    str    
 9   vaccinated  125 non-null    str    
 10  sterilized  125 non-null    str    
 11  owner_id    120 non-null    str    
dtypes: float64(1), str(11)
memory usage: 25.2 KB


### Типы данных

In [12]:
cats.dtypes

cat_id            str
name              str
breed             str
sex               str
color             str
birth_date        str
weight_kg         str
height_cm     float64
city              str
vaccinated        str
sterilized        str
owner_id          str
dtype: object

### Описательная статистика

In [13]:
cats.describe()

,height_cm
count,123.000000
mean,31.886179
std,3.383187
min,25.000000
25%,29.500000
50%,32.000000
75%,34.000000
max,42.000000


### Статистика по строковым столбцам

In [14]:
cats.describe(include='string')

,cat_id,name,breed,sex,color,birth_date,weight_kg,city,vaccinated,sterilized,owner_id
count,125,125,125,125,118,125,117,125,125,125,120
unique,120,29,17,2,10,119,54,8,6,2,81
top,CAT003,Кира,Бенгальская,кот,рыжий,2024-04-01,4.4,Томск,да,да,OWN011
freq,2,10,12,63,16,2,5,21,22,76,3


### Количество уникальных значений

In [15]:
cats.nunique().sort_values()

sex             2
sterilized      2
vaccinated      6
city            8
color          10
breed          17
height_cm      17
name           29
weight_kg      54
owner_id       81
birth_date    119
cat_id        120
dtype: int64

## 6. Выбор столбцов

In [16]:
cats['name'].head()

0      Лео
1      Лео
2    Злата
3      Мия
4     Кира
Name: name, dtype: str

In [18]:
cats[['name', 'breed', 'weight_kg', 'city']].head()

,name,breed,weight_kg,city
0,Лео,Мейн-кун,8.4,Томск
1,Лео,Британская короткошёрстная,4.4,Москва
2,Злата,Шотландская вислоухая,6.3,Москва
3,Мия,Сибирская,5.2,Санкт-Петербург
4,Кира,Сиамская,NaN,Екатеринбург


## 7. loc и iloc

In [19]:
cats.loc[0:4]

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021


In [20]:
cats.iloc[0:4]

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016


## 8. Фильтрация строк

In [23]:
cats[cats['city'] == 'Москва'].head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
17,CAT018,Персик,Канадский сфинкс,кошка,черепаховый,31.02.2021,4.0,30.0,Москва,НЕТ,да,OWN003
21,CAT022,Граф,Абиссинская,кошка,кремовый,2018-03-13,5.3,30.0,Москва,нет,нет,OWN023
27,CAT028,Граф,Сибирская,кошка,черепаховый,2022-08-31,7.1,35.0,Москва,нет,да,OWN053


### Несколько условий

In [26]:
moscow_female = cats[
    (cats['city'] == 'Москва') &
    (cats['sex'] == 'кошка')
]

moscow_female.head(10)

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
17,CAT018,Персик,Канадский сфинкс,кошка,черепаховый,31.02.2021,4.0,30.0,Москва,НЕТ,да,OWN003
21,CAT022,Граф,Абиссинская,кошка,кремовый,2018-03-13,5.3,30.0,Москва,нет,нет,OWN023
27,CAT028,Граф,Сибирская,кошка,черепаховый,2022-08-31,7.1,35.0,Москва,нет,да,OWN053
31,CAT032,Алиса,Персидская,кошка,колор-пойнт,2023-08-09,нет данных,29.0,Москва,Да,нет,OWN073
35,CAT036,Кузя,Норвежская лесная,кошка,голубой,2022-01-19,5.0,31.0,Москва,НЕТ,да,OWN010
41,CAT042,Симба,Канадский сфинкс,кошка,рыжий,2020-06-01,3.5,26.0,Москва,НЕТ,нет,OWN040
43,CAT044,Оскар,Персидская,кошка,чёрный,2016-10-11,5.8,35.0,Москва,Да,да,OWN050
45,CAT046,Снежок,Абиссинская,кошка,NaN,2022-09-27,2.9,27.0,Москва,нет,да,OWN060
51,CAT052,Снежок,Сибирская,кошка,серебристый,2017-06-04,6.1,33.0,Москва,нет,нет,OWN007


### isin()

In [27]:
cats[
    (cats['city'] == 'Москва') |
    (cats['city'] == 'Новосибирск') |
    (cats['city'] == 'Томск')
].head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
6,CAT007,Ириска,Бенгальская,кот,белый,2018-05-07,"4,9",32.0,Новосибирск,да,нет,OWN031


In [28]:
selected_cities = ['Москва', 'Новосибирск', 'Томск']

cats[cats['city'].isin(selected_cities)].head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
6,CAT007,Ириска,Бенгальская,кот,белый,2018-05-07,"4,9",32.0,Новосибирск,да,нет,OWN031


### query()

In [29]:
cats.query("city == 'Москва' and sex == 'кошка'").head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
17,CAT018,Персик,Канадский сфинкс,кошка,черепаховый,31.02.2021,4.0,30.0,Москва,НЕТ,да,OWN003
21,CAT022,Граф,Абиссинская,кошка,кремовый,2018-03-13,5.3,30.0,Москва,нет,нет,OWN023
27,CAT028,Граф,Сибирская,кошка,черепаховый,2022-08-31,7.1,35.0,Москва,нет,да,OWN053
31,CAT032,Алиса,Персидская,кошка,колор-пойнт,2023-08-09,нет данных,29.0,Москва,Да,нет,OWN073


## 9. Сортировка

In [ ]:
cats.sort_values('city')

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021
11,CAT012,Марта,Норвежская лесная,кошка,белый,2023-04-07,7.8,39.0,Екатеринбург,НЕТ,нет,OWN056
20,CAT021,Кира,Рэгдолл,кот,черепаховый,2024-05-17,5.0,32.0,Екатеринбург,ДА,да,OWN018
22,CAT023,Ириска,Русская голубая,кот,белый,2023-10-27,"6,2",33.0,Екатеринбург,Нет,да,OWN028
25,CAT026,Соня,британская короткошёрстная,кошка,колор-пойнт,2020-12-01,3.9,27.0,Екатеринбург,Да,да,OWN043
...,...,...,...,...,...,...,...,...,...,...,...,...
112,CAT113,Луна,Сиамская,кот,серый,2020-09-11,4.3,30.0,Томск,Нет,да,NaN
117,CAT118,Марс,Абиссинская,кошка,серый,2024-04-06,"4,5",28.0,Томск,нет,да,OWN005
118,CAT119,Тиша,Русская голубая,кот,черепаховый,2024-12-26,4.5,30.0,Томск,Нет,да,OWN010
122,CAT055,Мия,Бенгальская,кот,рыжий,2020-01-04,6.2,33.0,Томск,да,нет,OWN022


In [33]:
cats.sort_values(['city', 'weight_kg'], ascending=[True, False])

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
48,CAT049,Алиса,Мейн-кун,кот,серебристый,2025-03-09,9.1,39.0,Екатеринбург,да,да,OWN075
11,CAT012,Марта,Норвежская лесная,кошка,белый,2023-04-07,7.8,39.0,Екатеринбург,НЕТ,нет,OWN056
83,CAT084,Кузя,Норвежская лесная,кошка,колор-пойнт,2020-01-08,7.5,37.0,Екатеринбург,НЕТ,да,OWN001
42,CAT043,Граф,Бенгальская,кот,колор-пойнт,2021-12-02,7.3,38.0,Екатеринбург,да,да,OWN045
80,CAT081,Оскар,Рэгдолл,кот,серый,2019-06-11,7.3,38.0,Екатеринбург,ДА,да,OWN069
...,...,...,...,...,...,...,...,...,...,...,...,...
111,CAT112,Тиша,Сибирская,кошка,черепаховый,2024-04-29,4.5,30.0,Томск,нет,нет,OWN058
118,CAT119,Тиша,Русская голубая,кот,черепаховый,2024-12-26,4.5,30.0,Томск,Нет,да,OWN010
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
112,CAT113,Луна,Сиамская,кот,серый,2020-09-11,4.3,30.0,Томск,Нет,да,NaN


# Часть II. Очистка данных

Далее создадим отдельную копию таблицы, чтобы исходные данные всегда оставались доступными.


In [ ]:
cats_clean = cats.copy()
cats_clean.shape

(125, 12)

## 10. Поиск пропусков

In [36]:
cats_clean.isna().sum()

cat_id        0
name          0
breed         0
sex           0
color         7
birth_date    0
weight_kg     8
height_cm     2
city          0
vaccinated    0
sterilized    0
owner_id      5
dtype: int64

### Доля пропусков в процентах

In [37]:
cats_clean.isna().mean().mul(100).round().sort_values(ascending=False)

color         6.0
weight_kg     6.0
owner_id      4.0
height_cm     2.0
cat_id        0.0
name          0.0
breed         0.0
sex           0.0
birth_date    0.0
city          0.0
vaccinated    0.0
sterilized    0.0
dtype: float64

### Строки, в которых отсутствует вес

In [38]:
cats_clean[cats_clean['weight_kg'].isna()]

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021
19,CAT020,Соня,Персидская,кошка,кремовый,2023-12-18,NaN,30.0,Санкт-Петербург,Да,нет,OWN013
28,CAT029,Ириска,Сиамская,кот,колор-пойнт,2024-09-22,NaN,28.0,Санкт-Петербург,Нет,да,OWN058
46,CAT047,Злата,Русская голубая,кот,белый,2024-01-09,NaN,29.0,Красноярск,Нет,нет,OWN065
58,CAT059,Лео,Русская голубая,кот,голубой,2021-04-28,NaN,33.0,Новосибирск,Нет,да,OWN042
72,CAT073,Мия,Мейн-кун,кот,белый,2019-03-20,NaN,34.0,Санкт-Петербург,да,да,OWN029
91,CAT092,Злата,Персидская,кошка,чёрный,2024-07-23,NaN,31.0,Казань,Да,нет,OWN041
114,CAT115,Симба,Бенгальская,кот,чёрный,2020-01-30,NaN,30.0,Екатеринбург,да,нет,OWN073


## 11. Дубликаты

In [39]:
cats_clean.duplicated().sum()

np.int64(5)

### Удаляем дубликаты

In [41]:
cats_clean = cats_clean.drop_duplicates().copy()
cats_clean.shape

(120, 12)

## 12. Очистка текстовых значений

In [42]:
cats_clean['breed'].unique()

<ArrowStringArray>
[                  'Мейн-кун', 'Британская короткошёрстная',
      'Шотландская вислоухая',                  'Сибирская',
                   'Сиамская',           'Канадский сфинкс',
                'Бенгальская',                 'Персидская',
                 '  Рэгдолл ',                'Абиссинская',
            'Русская голубая',          'Норвежская лесная',
                    'Рэгдолл', 'британская короткошёрстная',
              '  Персидская ',                'абиссинская',
                '  Мейн-кун ']
Length: 17, dtype: str

### Убираем лишние пробелы

In [43]:
cats_clean['breed'] = cats_clean['breed'].str.strip()

cats_clean['breed'].unique()

<ArrowStringArray>
[                  'Мейн-кун', 'Британская короткошёрстная',
      'Шотландская вислоухая',                  'Сибирская',
                   'Сиамская',           'Канадский сфинкс',
                'Бенгальская',                 'Персидская',
                    'Рэгдолл',                'Абиссинская',
            'Русская голубая',          'Норвежская лесная',
 'британская короткошёрстная',                'абиссинская']
Length: 14, dtype: str

### Нормализуем регистр пород

In [44]:
cats_clean['breed'] = (
    cats_clean['breed']
    .str.strip()
    .str.lower()
)

cats_clean['breed'].value_counts()

breed
мейн-кун                      10
британская короткошёрстная    10
шотландская вислоухая         10
сибирская                     10
сиамская                      10
канадский сфинкс              10
бенгальская                   10
персидская                    10
рэгдолл                       10
абиссинская                   10
русская голубая               10
норвежская лесная             10
Name: count, dtype: int64

## 13. Нормализация признака vaccinated

In [45]:
cats_clean['vaccinated'] = (
    cats_clean['vaccinated']
    .str.strip()
    .str.lower()
)

cats_clean['vaccinated'].value_counts(dropna=False)

vaccinated
да     60
нет    60
Name: count, dtype: int64

## 14. Очистка веса

In [46]:
cats_clean['weight_kg'].head(20)

0     8.4
1     4.4
2     6.3
3     5.2
4     NaN
5     4.4
6     4,9
7     3.3
8     5.3
9     3.6
10    4.7
11    7.8
12    5.4
13    6.0
14    3.8
15    4.1
16    3.4
17    4.0
18    4.4
19    NaN
Name: weight_kg, dtype: str

In [47]:
cats_clean['weight_kg'] = (
    cats_clean['weight_kg']
    .astype('string')
    .str.strip()
    .str.replace(",", ".", regex=False)
)

cats_clean['weight_kg'].head(20)


0      8.4
1      4.4
2      6.3
3      5.2
4     <NA>
5      4.4
6      4.9
7      3.3
8      5.3
9      3.6
10     4.7
11     7.8
12     5.4
13     6.0
14     3.8
15     4.1
16     3.4
17     4.0
18     4.4
19    <NA>
Name: weight_kg, dtype: string

In [48]:
cats_clean['weight_kg'] = pd.to_numeric(
    cats_clean['weight_kg'],
    errors='coerce'
)

cats_clean['weight_kg'].head(20)

0      8.4
1      4.4
2      6.3
3      5.2
4     <NA>
5      4.4
6      4.9
7      3.3
8      5.3
9      3.6
10     4.7
11     7.8
12     5.4
13     6.0
14     3.8
15     4.1
16     3.4
17     4.0
18     4.4
19    <NA>
Name: weight_kg, dtype: Float64

In [49]:
cats_clean['weight_kg'].dtype

Float64Dtype()

In [50]:
cats_clean['weight_kg'].isna().sum()

np.int64(12)

In [51]:
median_weight = cats_clean['weight_kg'].median()

cats_clean['weight_kg'] = (
    cats_clean['weight_kg']
    .fillna(median_weight)
)

median_weight

np.float64(5.15)

## 15. Преобразование дат

In [52]:
cats_clean['birth_date'].head(20)

0     2021-11-24
1     2016-09-04
2     2024-04-01
3     2024-09-06
4     2023-09-27
5     2018-09-13
6     2018-05-07
7     2020-10-22
8     2016-08-13
9     2019-06-26
10    2019-03-23
11    2023-04-07
12    2022-03-30
13    2024-02-11
14    2020-12-08
15    2022-03-31
16    2023-12-06
17    31.02.2021
18    2024-04-12
19    2023-12-18
Name: birth_date, dtype: str

In [53]:
cats_clean['birth_date'] = pd.to_datetime(
    cats_clean['birth_date'],
    errors = 'coerce'
)

In [54]:
cats_clean['birth_date'].head(20)

0    2021-11-24
1    2016-09-04
2    2024-04-01
3    2024-09-06
4    2023-09-27
5    2018-09-13
6    2018-05-07
7    2020-10-22
8    2016-08-13
9    2019-06-26
10   2019-03-23
11   2023-04-07
12   2022-03-30
13   2024-02-11
14   2020-12-08
15   2022-03-31
16   2023-12-06
17          NaT
18   2024-04-12
19   2023-12-18
Name: birth_date, dtype: datetime64[us]

## 16. Создание новых признаков

In [55]:
cats_clean['weight_g'] = cats_clean['weight_kg'] * 1000
cats_clean[['weight_kg','weight_g']]

,weight_kg,weight_g
0,8.4,8400.0
1,4.4,4400.0
2,6.3,6300.0
3,5.2,5200.0
4,5.15,5150.0
...,...,...
115,5.5,5500.0
116,6.1,6100.0
117,4.5,4500.0
118,4.5,4500.0


In [56]:
cats_clean['is_large'] = cats_clean['weight_kg'] >= 6

## 17. map()

In [57]:
cats_clean['sex_code'] = cats_clean['sex'].map({
    'кот':'M',
    'кошка':'F'
})

cats_clean[['sex', 'sex_code']]

,sex,sex_code
0,кот,M
1,кошка,F
2,кот,M
3,кошка,F
4,кот,M
...,...,...
115,кошка,F
116,кот,M
117,кошка,F
118,кот,M


## 18. apply()

In [58]:
def weight_category(weight):
    if pd.isna(weight):
        return 'нет данных'
    if weight < 4:
        return 'легкий'
    elif weight < 6:
        return 'средний'
    else:
        return 'крупный'

cats_clean['weight_category'] = cats_clean['weight_kg'].apply(weight_category)

cats_clean['weight_category'].value_counts()

weight_category
средний    66
крупный    35
легкий     19
Name: count, dtype: int64

## 19. Векторизация вместо цикла

Для простых операций предпочтительнее использовать векторизованные выражения Pandas.


In [59]:
cats_clean['height_m'] = cats_clean['height_cm'] / 100
cats_clean[['height_m', 'height_cm']]

,height_m,height_cm
0,0.37,37.0
1,0.31,31.0
2,0.35,35.0
3,0.33,33.0
4,0.29,29.0
...,...,...
115,0.32,32.0
116,0.35,35.0
117,0.28,28.0
118,0.30,30.0


# Часть III. Анализ данных

## 20. value_counts()

### Нормализованные доли

## 21. Базовые агрегаты

## 22. GroupBy

### Несколько показателей

## 23. GroupBy по нескольким полям

## 24. pivot_table()

## 25. Справочник пород

## 26. merge(): cats + breeds

## 27. Таблица ветеринарных визитов

## 28. merge(): cats + visits

## 29. Количество визитов на кота

### Добавляем агрегаты визитов к котам

## 30. Анализ расходов по городам

## 31. Анализ причин визитов

## 32. Работа с временными данными

# Часть IV. Полезные практические приёмы

## 33. rename()

## 34. assign()

## 35. method chaining

## 36. Проверки качества данных

### Проверка диапазонов

## 37. Экспорт результата

# Часть V. Итоговый pipeline

Ниже — компактная версия полного сценария, которую можно адаптировать под реальные проекты.


## Итоговая аналитическая таблица

## Что изучено

В этом ноутбуке показаны:

- `read_csv()` и `read_excel()`;
- `Series`, `DataFrame`, `Index`;
- `head()`, `tail()`, `sample()`, `info()`, `describe()`;
- выбор столбцов, `loc`, `iloc`;
- фильтрация, `isin()`, `query()`;
- сортировка;
- строковые методы `.str`;
- пропуски и дубликаты;
- `to_numeric()` и `to_datetime()`;
- создание новых признаков;
- `map()` и `apply()`;
- `value_counts()`;
- `groupby()` и `agg()`;
- `pivot_table()`;
- `merge()` для `many-to-one` и `one-to-many`;
- анализ временных данных;
- method chaining;
- проверки качества;
- экспорт результатов.

Ноутбук можно использовать как основу для практического занятия или лабораторной работы.
